# Day 69: Convolutional Neural Networks (CNN) Basics

## Why CNNs for Images?
A plain fully-connected (Dense) network treats every pixel as an independent input and has no notion of spatial structure, so it needs a huge number of weights to learn even simple patterns, and it doesn't naturally recognize that the same shape (like an edge) can appear anywhere in the image. Convolutional Neural Networks (CNNs) solve this by sliding small learnable filters (also called kernels) across the image - each filter is reused at every position, so the same edge-detector, for example, works whether the edge is in the top-left or bottom-right of the image. This weight sharing dramatically reduces the number of parameters compared to a Dense layer of similar size, and it builds in the assumption that local pixel neighborhoods (not far-apart pixels) are what matter most for recognizing shapes, textures, and objects. This notebook builds up a CNN piece by piece - convolution, pooling, then flattening into a classifier - to show what each layer contributes.</cell id="cell-1">


In [ ]:
# pip install tensorflow
from tensorflow import keras
from tensorflow.keras import layers

## Convolution Layer
`Conv2D` is the core building block of a CNN: it slides a small grid of learnable weights (a filter/kernel) across the input image and, at each position, computes a weighted sum of the pixels under it to produce one output value. Using many filters (32 here) lets the layer learn many different local patterns simultaneously - some filters might learn to detect vertical edges, others horizontal edges or corners - and the output is a stack of "feature maps," one per filter, each showing where in the image that pattern was detected. `input_shape=(28, 28, 1)` describes a 28x28 grayscale image (1 color channel; RGB images would use 3), which matches the MNIST digit image format we'll use later. Running `model.summary()` prints the layer-by-layer shape of the data as it flows through the network along with the parameter count per layer, which is a quick way to sanity-check the architecture.</cell id="cell-3">


In [ ]:
model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),  # 32 filters, each a 3x3 sliding window; input is a 28x28 grayscale image
    layers.MaxPooling2D((2, 2)),  # downsamples each feature map by taking the max over 2x2 blocks
])
model.summary()  # shows output shape and parameter count for each layer so far

## Pooling Layer
`MaxPooling2D` shrinks each feature map by sliding a small window (2x2 here) across it and keeping only the maximum value in each window, which reduces the height and width by roughly half. This does two useful things: it cuts down the amount of computation and the number of parameters needed in later layers, and it makes the network a bit more tolerant to small shifts or distortions in where a feature appears (since we only care that the strongest activation was somewhere in that window, not its exact pixel position). Pooling has no learnable weights - it's a fixed downsampling operation, unlike the Conv2D layer's learned filters. Stacking Conv2D + MaxPooling2D blocks like this (as done again below with 64 filters) is the classic CNN pattern: each block extracts progressively higher-level features while reducing the spatial size of the data.</cell id="cell-5">


In [ ]:
model.add(layers.Conv2D(64, (3, 3), activation='relu'))  # second conv block, more filters (64) to learn more complex/combined patterns
model.add(layers.MaxPooling2D((2, 2)))  # downsample again
model.summary()

## Flatten & Dense Output
The convolution and pooling layers output a 3D stack of feature maps (height x width x number of filters), but a standard Dense (fully-connected) layer expects a 1D vector of inputs. `Flatten` simply reshapes that 3D block into a single long 1D vector without changing any values, which lets us feed the extracted image features into ordinary Dense layers for the final classification decision. From there it's the same pattern as any classifier: a Dense hidden layer with ReLU to combine the flattened features, followed by a Dense(10, softmax) output layer that produces a probability distribution over the 10 digit classes. This Conv/Pool -> ... -> Flatten -> Dense -> softmax structure is the standard shape of an image classification CNN.</cell id="cell-7">


In [ ]:
model.add(layers.Flatten())  # reshape the 3D feature maps into a 1D vector for the Dense layers
model.add(layers.Dense(64, activation='relu'))  # combines extracted features
model.add(layers.Dense(10, activation='softmax'))  # 10-class output probabilities
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()